In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import altair as alt
from vega_datasets import data
import time
import json

In [2]:
# Scrape taxation system table from International taxation page
import re

url1 = "https://en.wikipedia.org/wiki/International_taxation"
request_headers = {
    'User-Agent': 'Educational Project (contact: student@university.edu)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
}

response1 = requests.get(url1, headers=request_headers)
time.sleep(1)  # Be polite to Wikipedia servers

soup1 = BeautifulSoup(response1.content, 'html.parser')

# Find the taxation system table
tables1 = soup1.find_all('table', {'class': 'wikitable'})
print(f"Found {len(tables1)} tables on the International taxation page")

def extract_taxation_system(notes, taxes_foreign_citizens):
    """Extract the taxation system type from Notes and tax columns"""
    if pd.isna(notes) or notes == '':
        return 'Unknown'
    
    notes_lower = notes.lower()
    
    # Check for citizenship-based taxation FIRST (most specific)
    # This includes checking both notes and if the country taxes non-resident citizens
    if ('citizenship-based' in notes_lower or 'citizen-based' in notes_lower or 
        (taxes_foreign_citizens == 'Yes' and 'citizenship' in notes_lower)):
        return 'Citizenship-based'
    elif 'no personal income tax' in notes_lower:
        return 'No income tax'
    elif 'territorial taxation' in notes_lower:
        return 'Territorial'
    elif 'residential taxation' in notes_lower or 'residence-based' in notes_lower:
        return 'Residential'
    else:
        # Try to extract any phrase ending with "taxation"
        match = re.search(r'(\w+(?:\s+\w+)*)\s+taxation', notes_lower)
        if match:
            return match.group(1).title()
        return 'Other'

taxation_systems = []
if len(tables1) > 0:
    table = tables1[0]  # First wikitable on the page
    rows = table.find_all('tr')
    
    # Parse multi-level headers
    # Row 0: Main headers with colspan
    # Row 1: Sub-headers
    header_row1 = rows[0]
    header_row2 = rows[1] if len(rows) > 1 else None
    
    # Get main headers
    main_headers = header_row1.find_all('th')
    
    # Build column names considering colspan
    columns = []
    for th in main_headers:
        text = th.get_text(strip=True)
        colspan = int(th.get('colspan', 1))
        
        if colspan > 1 and header_row2:
            # Get subheaders for this main header
            sub_headers = header_row2.find_all(['th', 'td'])
            for j in range(colspan):
                if len(sub_headers) > len(columns):
                    sub_text = sub_headers[len(columns)].get_text(strip=True)
                    columns.append(f"{text} {sub_text}".strip())
        else:
            columns.append(text)
    
    print(f"Table columns: {columns}")
    
    # Process data rows - skip first 2 rows (headers)
    for row in rows[2:]:
        cells = row.find_all(['td', 'th'])
        if len(cells) >= 4:  # Need at least country + 3 tax columns
            country = cells[0].get_text(strip=True)
            
            # Only add valid country entries
            if country and len(country) > 2 and country.lower() not in ['country', 'jurisdiction', 'territory']:
                notes = cells[4].get_text(strip=True) if len(cells) > 4 else ''
                taxes_foreign_citizens = cells[3].get_text(strip=True) if len(cells) > 3 else ''
                
                row_data = {
                    'Country': country,
                    'Taxes local income': cells[1].get_text(strip=True) if len(cells) > 1 else '',
                    'Taxes foreign income of residents': cells[2].get_text(strip=True) if len(cells) > 2 else '',
                    'Taxes foreign income of non-resident citizens': taxes_foreign_citizens,
                    'Notes': notes,
                    'Taxation System': extract_taxation_system(notes, taxes_foreign_citizens)
                }
                taxation_systems.append(row_data)

df_systems = pd.DataFrame(taxation_systems)
print(f"\nScraped {len(df_systems)} countries with taxation systems")
print(f"Columns: {list(df_systems.columns)}")
print(f"\nTaxation systems found:")
print(df_systems['Taxation System'].value_counts())
df_systems.head(10)


Found 1 tables on the International taxation page
Table columns: ['Country or territory', 'Taxes local income', 'Notes and sources']

Scraped 238 countries with taxation systems
Columns: ['Country', 'Taxes local income', 'Taxes foreign income of residents', 'Taxes foreign income of non-resident citizens', 'Notes', 'Taxation System']

Taxation systems found:
Taxation System
Residential          169
Territorial           45
No income tax         19
Citizenship-based      5
Name: count, dtype: int64


,Country,Taxes local income,Taxes foreign income of residents,Taxes foreign income of non-resident citizens,Notes,Taxation System
0,Antigua and Barbuda,No,No,No,No personal income tax.[4][5],No income tax
1,Bahamas,No,No,No,No personal income tax.[6],No income tax
2,Bahrain,No,No,No,No personal income tax.[6],No income tax
3,Brunei,No,No,No,No personal income tax.[6],No income tax
4,Cayman Islands,No,No,No,No personal income tax.[6],No income tax
5,Kuwait,No,No,No,No personal income tax.[6],No income tax
6,Monaco,No,No,No,No personal income tax.[7],No income tax
7,Oman,No,No,No,No personal income tax.[6],No income tax
8,Pitcairn Islands,No,No,No,No personal income tax.[8],No income tax
9,Qatar,No,No,No,No personal income tax.[6],No income tax


In [3]:
# Scrape tax rates by countries table
url2 = "https://en.wikipedia.org/wiki/List_of_countries_by_tax_rates"

response2 = requests.get(url2, headers=request_headers)
time.sleep(1)  # Be polite to Wikipedia servers

soup2 = BeautifulSoup(response2.content, 'html.parser')

# Find the main table with tax rates
tax_rates = []
table = soup2.find('table', {'class': 'wikitable'})

if table:
    rows = table.find_all('tr')
    
    # Parse multi-level headers
    # Row 0: Main headers (Individual income has colspan=2)
    # Row 1: Sub-headers (Lowest/Highest for Individual income)
    header_row1 = rows[0].find_all(['th', 'td'])
    header_row2 = rows[1].find_all(['th', 'td'])
    
    # Build column names properly handling colspan and rowspan
    columns = []
    sub_idx = 0
    
    for th in header_row1:
        text = th.get_text(strip=True)
        colspan = int(th.get('colspan', 1))
        rowspan = int(th.get('rowspan', 1))
        
        if colspan > 1:
            # This column has subheaders (e.g., "Individual income" with Lowest/Highest)
            for j in range(colspan):
                if sub_idx < len(header_row2):
                    sub_text = header_row2[sub_idx].get_text(strip=True)
                    columns.append(f"{text} {sub_text}".strip())
                    sub_idx += 1
        else:
            # Single column (may span multiple rows)
            columns.append(text)
    
    print(f"Table columns ({len(columns)}): {columns}")
    
    # Process data rows - skip first 2 rows (headers)
    for row in rows[2:]:
        cells = row.find_all(['td', 'th'])
        if len(cells) >= len(columns):
            country = cells[0].get_text(strip=True)
            
            # Only add valid country entries
            if country and len(country) > 2 and country.lower() not in ['country', 'jurisdiction', 'lowest', 'highest']:
                row_data = {}
                for i, col_name in enumerate(columns):
                    row_data[col_name] = cells[i].get_text(strip=True) if i < len(cells) else ''
                
                tax_rates.append(row_data)

df_tax_rates = pd.DataFrame(tax_rates)
print(f"\nScraped {len(df_tax_rates)} countries with tax rate information")
print(f"Columns: {list(df_tax_rates.columns)}")
df_tax_rates.head(10)


Table columns (10): ['Tax jurisdiction', 'Corporate', 'Individual income Lowest', 'Individual income Highest', 'Capital gains[1]', 'Wealth', 'Property', 'Inheritance/Estate', 'VAT or GSTorSales', 'Further reading']

Scraped 207 countries with tax rate information
Columns: ['Tax jurisdiction', 'Corporate', 'Individual income Lowest', 'Individual income Highest', 'Capital gains[1]', 'Wealth', 'Property', 'Inheritance/Estate', 'VAT or GSTorSales', 'Further reading']


,Tax jurisdiction,Corporate,Individual income Lowest,Individual income Highest,Capital gains[1],Wealth,Property,Inheritance/Estate,VAT or GSTorSales,Further reading
0,Afghanistan,20%[2],0%[3],20%[3],,,,,"0%(however, in Taliban run areas pre-Taliban r...",Taxation in Afghanistan
1,Albania,15%[6],0%[7],23%[7],15%,,,,20%(standard)6%(tourism services)[8],Taxation in Albania
2,Algeria,19–26%[9],0%[10],35%[10],15%(resident)20%(non-resident),,,,19%(standard)[11]9%(basic items)[11],Taxation in Algeria
3,American Samoa,34%[6],4%[12][13],6%[13],,,0%,,0%[14][13],Taxation in American Samoa
4,Andorra,10%[15],0%[16],10%[16],,,,,"4.5%(standard)9.5%(banking services)2.5%, 1% o...",Taxation in Andorra
5,Angola,30%[18],0%[18],17%[18],10%,,,,14%[19],Taxation in Angola
6,Argentina,35%(residents)15%(non-residents)[24],9%[24],35%[24],15%,,,,21%[24],Taxation in Argentina
7,Armenia,18%[25],22%[25],22%[25],10–20%,,,,20%[25],Taxation in Armenia
8,Aruba,25%[26],7%[26],58.95%[26],,,,,1.5%(turnover tax)[26],Taxation in Aruba
9,Australia,30%(standard)25%(base entity)[27][Note 1],0%[27],45%[28][Note 2],0–45%,No[29],,0%,10%(standard)0%(essential items)[27],Taxation in Australia


In [4]:
# Clean and standardize country names for merging
def clean_country_name(name):
    # Remove references and extra characters
    name = name.split('[')[0].strip()
    name = name.split('(')[0].strip()
    return name

# Clean country names in both datasets
if 'Country' in df_systems.columns:
    df_systems['Country_clean'] = df_systems['Country'].apply(clean_country_name)
else:
    print("Warning: 'Country' column not found in taxation systems dataframe")

# Use 'Tax jurisdiction' as the country column
if 'Tax jurisdiction' in df_tax_rates.columns:
    df_tax_rates['Country_clean'] = df_tax_rates['Tax jurisdiction'].apply(clean_country_name)
else:
    print("Warning: Tax jurisdiction column not found in tax rates dataframe")
    print(f"Available columns: {list(df_tax_rates.columns)}")

# Merge the two datasets
df_merged = pd.merge(
    df_tax_rates,
    df_systems,
    on='Country_clean',
    how='left'
)

print(f"\nMerged dataset has {len(df_merged)} rows")
print(f"Columns in merged dataset: {list(df_merged.columns)}")
print(f"Countries with 'Taxes local income' info: {df_merged['Taxes local income'].notna().sum()}")
df_merged.head(10)



Merged dataset has 207 rows
Columns in merged dataset: ['Tax jurisdiction', 'Corporate', 'Individual income Lowest', 'Individual income Highest', 'Capital gains[1]', 'Wealth', 'Property', 'Inheritance/Estate', 'VAT or GSTorSales', 'Further reading', 'Country_clean', 'Country', 'Taxes local income', 'Taxes foreign income of residents', 'Taxes foreign income of non-resident citizens', 'Notes', 'Taxation System']
Countries with 'Taxes local income' info: 201


,Tax jurisdiction,Corporate,Individual income Lowest,Individual income Highest,Capital gains[1],Wealth,Property,Inheritance/Estate,VAT or GSTorSales,Further reading,Country_clean,Country,Taxes local income,Taxes foreign income of residents,Taxes foreign income of non-resident citizens,Notes,Taxation System
0,Afghanistan,20%[2],0%[3],20%[3],,,,,"0%(however, in Taliban run areas pre-Taliban r...",Taxation in Afghanistan,Afghanistan,Afghanistan,Yes,Yes,No,Residence-based taxation.[6],Residential
1,Albania,15%[6],0%[7],23%[7],15%,,,,20%(standard)6%(tourism services)[8],Taxation in Albania,Albania,Albania,Yes,Yes,No,Residence-based taxation.[6],Residential
2,Algeria,19–26%[9],0%[10],35%[10],15%(resident)20%(non-resident),,,,19%(standard)[11]9%(basic items)[11],Taxation in Algeria,Algeria,Algeria,Yes,Yes,No,Residence-based taxation.[6],Residential
3,American Samoa,34%[6],4%[12][13],6%[13],,,0%,,0%[14][13],Taxation in American Samoa,American Samoa,American Samoa,Yes,Yes,No,Residence-based taxation.[48],Residential
4,Andorra,10%[15],0%[16],10%[16],,,,,"4.5%(standard)9.5%(banking services)2.5%, 1% o...",Taxation in Andorra,Andorra,Andorra,Yes,Yes,No,Residence-based taxation.[49],Residential
5,Angola,30%[18],0%[18],17%[18],10%,,,,14%[19],Taxation in Angola,Angola,Angola,Yes,No,No,Territorial taxation.[6],Territorial
6,Argentina,35%(residents)15%(non-residents)[24],9%[24],35%[24],15%,,,,21%[24],Taxation in Argentina,Argentina,Argentina,Yes,Yes,No,Residence-based taxation.[6],Residential
7,Armenia,18%[25],22%[25],22%[25],10–20%,,,,20%[25],Taxation in Armenia,Armenia,Armenia,Yes,Yes,No,Residence-based taxation.[6],Residential
8,Aruba,25%[26],7%[26],58.95%[26],,,,,1.5%(turnover tax)[26],Taxation in Aruba,Aruba,Aruba,Yes,Yes,No,Residence-based taxation.[6],Residential
9,Australia,30%(standard)25%(base entity)[27][Note 1],0%[27],45%[28][Note 2],0–45%,No[29],,0%,10%(standard)0%(essential items)[27],Taxation in Australia,Australia,"Australia(includingChristmas Island,Cocos (Kee...",Yes,Yes*,No,Residence-based taxation.[6]* Except temporary...,Residential


In [5]:
# Prepare data for visualization with country name mapping
# Reimport vega_datasets to avoid variable conflict
from vega_datasets import data as vega_data

# Get world map data
countries = alt.topo_feature(vega_data.world_110m.url, 'countries')

# Create data for visualization
df_viz = df_merged.copy()

# Map country names to match the world map naming conventions
country_mapping = {
    'United States': 'United States of America',
    'United Kingdom': 'United Kingdom',
    'Russia': 'Russia',
    'South Korea': 'South Korea',
    'North Korea': 'North Korea',
    'DR Congo': 'Dem. Rep. Congo',
    'Congo': 'Congo',
    'Tanzania': 'Tanzania',
    'Ivory Coast': "Côte d'Ivoire",
    'Czech Republic': 'Czechia',
    'Timor-Leste': 'Timor-Leste',
    'Eswatini': 'eSwatini',
    'Serbia': 'Serbia',
    'Bosnia and Herzegovina': 'Bosnia and Herz.',
    'Dominican Republic': 'Dominican Rep.',
    'Central African Republic': 'Central African Rep.',
    'South Sudan': 'S. Sudan',
    'Solomon Islands': 'Solomon Is.',
    'Equatorial Guinea': 'Eq. Guinea',
    'Guinea-Bissau': 'Guinea-Bissau',
    'W. Sahara': 'W. Sahara'
}

# Apply the mapping
df_viz['name'] = df_viz['Country_clean'].replace(country_mapping)

# Fill missing taxation systems with "Unknown"
if 'Taxation System' in df_viz.columns:
    df_viz['Taxation System'] = df_viz['Taxation System'].fillna('Unknown')
else:
    df_viz['Taxation System'] = 'Unknown'

# Rename columns with simpler names for Altair compatibility
df_viz = df_viz.rename(columns={
    'Individual income Lowest': 'Income_Tax_Min',
    'Individual income Highest': 'Income_Tax_Max',
    'VAT or GSTorSales': 'VAT_GST_Sales'
})

# Select only the columns we need for visualization to keep data clean
viz_columns = ['name', 'Taxation System', 'Corporate', 'Income_Tax_Min', 'Income_Tax_Max', 'VAT_GST_Sales']
df_viz_clean = df_viz[viz_columns].copy()

print(f"Prepared {len(df_viz_clean)} countries for visualization")
print(f"Taxation systems found: {df_viz_clean['Taxation System'].unique()}")
print(f"Columns: {list(df_viz_clean.columns)}")
print(f"\nSample data:")
print(df_viz_clean.head())


Prepared 207 countries for visualization
Taxation systems found: ['Residential' 'Territorial' 'Unknown' 'No income tax' 'Citizenship-based']
Columns: ['name', 'Taxation System', 'Corporate', 'Income_Tax_Min', 'Income_Tax_Max', 'VAT_GST_Sales']

Sample data:
             name Taxation System  Corporate Income_Tax_Min Income_Tax_Max  \
0     Afghanistan     Residential     20%[2]          0%[3]         20%[3]   
1         Albania     Residential     15%[6]          0%[7]         23%[7]   
2         Algeria     Residential  19–26%[9]         0%[10]        35%[10]   
3  American Samoa     Residential     34%[6]     4%[12][13]         6%[13]   
4         Andorra     Residential    10%[15]         0%[16]        10%[16]   

                                       VAT_GST_Sales  
0  0%(however, in Taliban run areas pre-Taliban r...  
1               20%(standard)6%(tourism services)[8]  
2               19%(standard)[11]9%(basic items)[11]  
3                                         0%[14][13] 

In [6]:
# Create an interactive rotating globe with Altair
alt.data_transformers.enable('default')

# Load world map with country names
world_url = 'https://cdn.jsdelivr.net/npm/world-atlas@2/countries-110m.json'
world_countries = alt.topo_feature(world_url, 'countries')

# Create interactive rotation parameters
rotate0 = alt.param(name='rotate0', value=0, bind=alt.binding_range(min=-180, max=180, step=1))
rotate1 = alt.param(name='rotate1', value=0, bind=alt.binding_range(min=-90, max=90, step=1))

# Sphere background (ocean)
sphere = alt.Chart(alt.sphere()).mark_geoshape(fill='aliceblue')

# Custom color scheme with royal blue
color_scheme = {
    'Citizenship-based': '#4169E1',  # Royal blue
    'Residential': '#FF9F80',         # Soft coral orange
    'Territorial': '#32CD32',         # Lime green
    'No income tax': '#FFD700',       # Gold
    'Unknown': '#D3D3D3',             # Light gray
    'Other': '#9370DB'                # Medium purple
}

# Countries layer with taxation data
countries_layer = alt.Chart(world_countries).mark_geoshape(
    stroke='white',
    strokeWidth=0.5
).transform_lookup(
    lookup='properties.name',
    from_=alt.LookupData(
        df_viz_clean, 
        'name', 
        ['name', 'Taxation System', 'Corporate', 'Income_Tax_Min', 'Income_Tax_Max', 'VAT_GST_Sales']
    )
).encode(
    color=alt.condition(
        alt.datum['Taxation System'] != None,
        alt.Color('Taxation System:N',
                  scale=alt.Scale(domain=list(color_scheme.keys()), 
                                 range=list(color_scheme.values())),
                  legend=alt.Legend(title='Taxation System')),
        alt.value('mintcream')
    ),
    tooltip=[
        alt.Tooltip('properties.name:N', title='Country'),
        alt.Tooltip('Taxation System:N', title='Taxation System'),
        alt.Tooltip('Corporate:N', title='Corporate Tax'),
        alt.Tooltip('Income_Tax_Min:N', title='Income Tax (Min)'),
        alt.Tooltip('Income_Tax_Max:N', title='Income Tax (Max)'),
        alt.Tooltip('VAT_GST_Sales:N', title='VAT/GST/Sales Tax')
    ]
)

# Combine layers with orthographic projection using expression for rotation
chart = alt.layer(sphere, countries_layer).add_params(
    rotate0, rotate1
).project(
    type='orthographic',
    rotate={'expr': '[rotate0, rotate1, 0]'}
).properties(
    width=600,
    height=600,
    title={
        'text': 'Interactive Globe: World Taxation Systems and Tax Rates',
        'subtitle': 'Use the sliders to rotate the globe and explore taxation systems globally',
        'anchor': 'start'
    }
).configure_view(
    stroke=None
)

chart


alt.LayerChart(...)

In [7]:
# Save the chart as JSON to the graphs folder
output_path = r'c:\Users\juanx\Documents\GitHub\juanxgi83.github.io\graphs\world_taxation_map.json'
chart.save(output_path)
print(f"Chart saved to: {output_path}")

Chart saved to: c:\Users\juanx\Documents\GitHub\juanxgi83.github.io\graphs\world_taxation_map.json


In [8]:
# Verify the data - check a few sample countries
print("Sample data verification:")
sample_countries = ['United States of America', 'United Kingdom', 'France', 'Germany', 'Japan', 'Brazil']
for country in sample_countries:
    data = df_viz[df_viz['name'] == country]
    if not data.empty:
        print(f"{country}: Tax System = {data['Taxation System'].values[0]}, Corporate = {data['Corporate'].values[0]}")
    else:
        print(f"{country}: NOT FOUND in dataset")

Sample data verification:
United States of America: Tax System = Citizenship-based, Corporate = 21%(federal)up to 21%(with credit of tax paid towards other countries)
United Kingdom: Tax System = Residential, Corporate = 19–25%[318]
France: Tax System = Residential, Corporate = 25%
Germany: Tax System = Residential, Corporate = 30%(15% corporate tax (+ 5.5% solidarity surcharge) + 7% to 17% trade tax)[133]
Japan: Tax System = Residential, Corporate = 29.74%[162]
Brazil: Tax System = Residential, Corporate = 40%(highest rate for financial institutions, insurance and capitalisation companies)24–34%(general)15%(+10% in profits exceeding BR$ 20.000[59]+ 9% Social Contribution Tax or 15% for financial institutions, insurance and capitalisation companies[60])


In [9]:
# Verify the final visualization data
print("Sample countries in the visualization:\n")
sample_df = df_viz_clean[['name', 'Taxation System', 'Corporate', 'Income_Tax_Min', 'Income_Tax_Max', 'VAT_GST_Sales']].head(15)
print(sample_df.to_string())

print("\n\nTaxation systems breakdown:")
print(df_viz_clean['Taxation System'].value_counts())

print("\n\nChart has 207 countries with complete data embedded in the JSON file.")


Sample countries in the visualization:

              name Taxation System                                                         Corporate                                                     Income_Tax_Min                                                           Income_Tax_Max                                                                                           VAT_GST_Sales
0      Afghanistan     Residential                                                            20%[2]                                                              0%[3]                                                                   20%[3]  0%(however, in Taliban run areas pre-Taliban rule, small fees were illegally added to groceries)[4][5]
1          Albania     Residential                                                            15%[6]                                                              0%[7]                                                                   23%[7]                              